# 01 — Data acquisition and verification

Verifies the five DepMap 26Q1 files already on disk. **Downloads nothing when they are
present** — they were copied read-only from the Aim 2 project and sha256-verified against
the originals.

Checks:
1. every file present, read-only, sha256 matching `src.utils.EXPECTED_SHA256`
2. the six primary genes plus WRAP53 present in `CRISPRGeneEffect.csv`, with **zero
   missing values** across the osteosarcoma lines
3. WRAP53 and TP53 present in the expression matrix

### If a file were ever missing

Re-fetch from the DepMap **26Q1 archive specifically** (`depmap.org/portal/download/all/`,
filter release = 26Q1), never from "latest": the frozen ALT labels were derived against
26Q1 and a different release would silently break the label join. Then update
`EXPECTED_SHA256` deliberately, as a reviewed change.

Reads **no gene-effect values** — availability and missingness only.

## 1 — Header

In [1]:
import sys, random, time, warnings
from pathlib import Path

import numpy as np

random.seed(42)
np.random.seed(42)

_cwd = Path('.').resolve()
ROOT = _cwd if (_cwd / 'src').is_dir() else _cwd.parent
assert (ROOT / 'src').is_dir(), f"Cannot locate src/ -- tried {_cwd} and {_cwd.parent}"
if str(ROOT) not in sys.path:
    sys.path.insert(0, str(ROOT))
print(f'ROOT: {ROOT}')

warnings.filterwarnings('ignore', category=FutureWarning)

import pandas as pd
from src.utils import log_versions, run_timestamp, record_provenance, write_report, write_sha256_manifest

RUN_TIMESTAMP = run_timestamp()
VERSIONS = log_versions(['pandas', 'numpy', 'scipy', 'sklearn', 'matplotlib'])
print(f'Run: {RUN_TIMESTAMP}')
assert sys.version_info[:2] == (3, 12), f'Expected Python 3.12, got {sys.version_info[:2]}'
print(f'Python: {sys.version.split()[0]}')

ROOT: /Users/rinatrizvanov/Desktop/Projects/Project 1 - Comp Bio for Osteosarcoma/tcab1-bir-dependency


Library versions:
  pandas: 2.3.3
  numpy: 2.5.4
  scipy: 1.18.1
  sklearn: 1.9.1
  matplotlib: 3.11.2
Run: 2026-10-12T01:23:13Z
Python: 3.12.13


## 2 — File integrity

In [2]:
from src.utils import (
    CRISPR_PATH, RNA_PATH, MODEL_PATH, CN_GENE_PATH, CN_SEGMENTS_PATH,
    DEPMAP_RELEASE, EXPECTED_SHA256, verify_expected,
)

PATHS = {
    'CRISPRGeneEffect.csv': CRISPR_PATH,
    'OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv': RNA_PATH,
    'Model.csv': MODEL_PATH,
    'OmicsCNGeneWGS.csv': CN_GENE_PATH,
    'OmicsCNSegmentsWGS.csv': CN_SEGMENTS_PATH,
}
print(f'DepMap release: {DEPMAP_RELEASE}\n')
rows = []
for name, path in PATHS.items():
    if not path.exists():
        raise FileNotFoundError(
            f'{name} is missing. Re-fetch from the DepMap {DEPMAP_RELEASE} archive '
            f'specifically (see this notebook header), not from "latest".'
        )
    res = verify_expected(path)       # raises on sha256 mismatch
    rows.append({
        'file': name,
        'size_mb': round(res['size_bytes'] / 1e6, 1),
        'sha256_match': res['checksum_match'],
        'read_only': not (path.stat().st_mode & 0o222),
    })
files = pd.DataFrame(rows)
print(files.to_string(index=False))
assert files['sha256_match'].all()
assert files['read_only'].all(), 'DepMap inputs must stay read-only'
print(f"\nAll {len(files)} files verified against EXPECTED_SHA256.")

DepMap release: 26Q1



                                              file  size_mb  sha256_match  read_only
                              CRISPRGeneEffect.csv    440.6          True       True
OmicsExpressionTPMLogp1HumanProteinCodingGenes.csv    305.0          True       True
                                         Model.csv      0.7          True       True
                                OmicsCNGeneWGS.csv    409.0          True       True
                            OmicsCNSegmentsWGS.csv     63.7          True       True

All 5 files verified against EXPECTED_SHA256.


## 3 — Gene availability and missingness

In [3]:
from src.cohort import osteosarcoma_models, crispr_model_ids, rna_model_ids
from src.dependency_stats import SIX_GENES, crispr_gene_columns
from src.stratify import TCAB1_GENE, TP53_GENE, expression_gene_columns

os_models = osteosarcoma_models()
screened = crispr_model_ids()
expressed = rna_model_ids()
os_ids = set(os_models['ModelID'])

print(f'osteosarcoma lines in Model.csv      : {len(os_models)}')
print(f'  with a CRISPR screen               : {len(os_ids & screened)}')
print(f'  in the expression matrix           : {len(os_ids & expressed)}')
print(f'  with both                          : {len(os_ids & screened & expressed)}')

# Column presence (raises naming any absent gene).
crispr_cols = crispr_gene_columns([*SIX_GENES, TCAB1_GENE])
expr_cols = expression_gene_columns([TCAB1_GENE, TP53_GENE])
print(f'\nCRISPR columns resolved  : {crispr_cols}')
print(f'Expression columns resolved: {expr_cols}')

osteosarcoma lines in Model.csv      : 35
  with a CRISPR screen               : 26
  in the expression matrix           : 34
  with both                          : 26

CRISPR columns resolved  : {'POLD3': 'POLD3 (10714)', 'RAD52': 'RAD52 (5893)', 'SMARCAL1': 'SMARCAL1 (50485)', 'BRCA1': 'BRCA1 (672)', 'BRCA2': 'BRCA2 (675)', 'RAD51': 'RAD51 (5888)', 'WRAP53': 'WRAP53 (55135)'}
Expression columns resolved: {'WRAP53': 'WRAP53 (55135)', 'TP53': 'TP53 (7157)'}


In [4]:
# Missingness across osteosarcoma lines -- counts only, no values retained.
import csv

targets = {**crispr_cols}
with open(CRISPR_PATH) as fh:
    header = next(fh).rstrip('\n').split(',')
    idx = {g: header.index(c) for g, c in targets.items()}
    n_lines, missing = 0, {g: 0 for g in targets}
    for line in fh:
        fields = line.rstrip('\n').split(',')
        if fields[0] not in os_ids:
            continue
        n_lines += 1
        for gene, i in idx.items():
            if fields[i] == '':
                missing[gene] += 1

avail = pd.DataFrame({
    'gene': list(missing),
    'missing_in_os_lines': list(missing.values()),
    'n_os_lines_screened': n_lines,
})
print(avail.to_string(index=False))
assert n_lines == 26, f'Expected 26 screened osteosarcoma lines, got {n_lines}'
assert all(v == 0 for v in missing.values()), f'Missing gene effects: {missing}'
print('\nAll primary genes present with zero missing values across the 26 screened lines.')

    gene  missing_in_os_lines  n_os_lines_screened
   POLD3                    0                   26
   RAD52                    0                   26
SMARCAL1                    0                   26
   BRCA1                    0                   26
   BRCA2                    0                   26
   RAD51                    0                   26
  WRAP53                    0                   26

All primary genes present with zero missing values across the 26 screened lines.


## 4 — Provenance document and report

In [5]:
from src.utils import DOCS_DIR, file_mtime_iso

prov = [
    '# Data Provenance',
    '',
    f'DepMap release: **{DEPMAP_RELEASE}**. Files copied read-only from the Aim 2 project',
    'and sha256-verified byte-identical to the originals.',
    '',
    '| File | Release | sha256 | Bytes |',
    '|---|---|---|---|',
]
# Deliberately no mtime column: mtimes are filesystem metadata that copying or
# cloning changes, so including them would make this document differ between
# faithful reproductions. The sha256 is the provenance that matters. Per-run
# mtimes go to provenance.jsonl.
for name, path in PATHS.items():
    prov.append(
        f'| `data/raw/depmap/{name}` | {DEPMAP_RELEASE} | `{EXPECTED_SHA256[name][:16]}...` '
        f'| {path.stat().st_size} |'
    )
prov += [
    '',
    '## Re-fetch policy',
    '',
    f'If a file must be replaced, fetch it from the DepMap **{DEPMAP_RELEASE} archive',
    'specifically**, never from "latest". The frozen ALT labels in',
    '`references/master_manifest.csv` were derived against this release; a different',
    'release changes which lines are screened and would silently break the label join.',
    '',
    '## Frozen label input',
    '',
    '| File | sha256 | Role |',
    '|---|---|---|',
    f"| `references/master_manifest.csv` | `{EXPECTED_SHA256['master_manifest.csv'][:16]}...` "
    '| ALT labels (`alt_call_direction`, `alt_status`); never re-derived |',
]
(DOCS_DIR / 'data_provenance.md').write_text('\n'.join(prov) + '\n')
print(f'wrote docs/data_provenance.md')

wrote docs/data_provenance.md


In [6]:
lines = [
    '# Notebook 01 — Data acquisition and verification',
    '',
    f'DepMap release **{DEPMAP_RELEASE}**. All inputs already on disk; nothing downloaded.',
    '',
    '## File integrity',
    '',
    '| File | MB | sha256 match | Read-only |',
    '|---|---|---|---|',
]
for _, r in files.iterrows():
    lines.append(f"| {r['file']} | {r['size_mb']} | {'yes' if r['sha256_match'] else 'NO'} "
                 f"| {'yes' if r['read_only'] else 'NO'} |")
lines += [
    '',
    '## Cohort availability',
    '',
    f'- osteosarcoma lines in `Model.csv`: {len(os_models)}',
    f'- with a CRISPR screen: {len(os_ids & screened)}',
    f'- in the expression matrix: {len(os_ids & expressed)}',
    f'- with both: {len(os_ids & screened & expressed)}',
    '',
    '## Gene availability (outcome genes + stratifier)',
    '',
    '| Gene | Missing across the 26 screened osteosarcoma lines |',
    '|---|---|',
]
for _, r in avail.iterrows():
    lines.append(f"| {r['gene']} | {r['missing_in_os_lines']} |")
lines += [
    '',
    'No imputation is needed or permitted: every primary gene is fully observed.',
    '',
    '## Note',
    '',
    'This notebook reads gene-effect *availability* only. No gene-effect value is',
    'retained or inspected; the outcome is first loaded in notebook 05, after the',
    'pre-specification freeze.',
]
report = write_report('01', lines)
manifest = write_sha256_manifest('01', [report, DOCS_DIR / 'data_provenance.md'])
record_provenance([{
    'notebook': '01', 'run_timestamp': RUN_TIMESTAMP, 'depmap_release': DEPMAP_RELEASE,
    'files_verified': int(len(files)), 'n_os_screened': int(n_lines),
    'input_mtimes': {name: file_mtime_iso(path) for name, path in PATHS.items()},
    'missing_gene_effects': int(sum(missing.values())),
}])
print(f'wrote {report.name}')
print(f'wrote {manifest.name}')

wrote notebook_01_report.md
wrote notebook_01_sha256.json
